# Polarization example - maximum likelihood method

This notebook fits the polarization fraction and angle of a Data Challenge 3 GRB (GRB 080802386) simulated using MEGAlib and combined with albedo photon background. It's assumed that the start time, duration, localization, and spectrum of the GRB are already known. The GRB was simulated with 80% polarization at an angle of 90 degrees in the IAU convention, and was 20 degrees off-axis. 

In [ ]:
from cosipy import UnBinnedData, COSILike
from cosipy.spacecraftfile import SpacecraftFile
from cosipy.threeml.custom_functions import Band_Eflux
from astropy.time import Time
import numpy as np
from astropy.coordinates import SkyCoord
from astropy import units as u
from scoords import SpacecraftFrame
from cosipy.util import fetch_wasabi_file
from pathlib import Path
from threeML import LinearPolarization, SpectralComponent, PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter

### Download and read in data

This will download the files needed to run this notebook. If you have already downloaded these files, you can skip this.

Download the unbinned data (660.58 KB)

In [ ]:
fetch_wasabi_file('COSI-SMEX/cosipy_tutorials/polarization_fit/grb_background.fits.gz', checksum = '21b1d75891edc6aaf1ff3fe46e91cb49')

Download the polarization response (217.47 MB)

In [ ]:
fetch_wasabi_file('COSI-SMEX/develop/Data/Responses/ResponseContinuum.o3.pol.e200_10000.b4.p12.relx.s10396905069491.m420.filtered.binnedpolarization.11D.h5', checksum = '46b006a6b397fd777dc561d3b028357f')

Download the orientation file (1.10 GB)

In [ ]:
fetch_wasabi_file('COSI-SMEX/DC3/Data/Orientation/DC3_final_530km_3_month_with_slew_1sbins_GalacticEarth_SAA.ori', checksum = 'b87fd41b6c28a5c0c51448ce2964e57c')

Read in and bin the data, which is a GRB placed within albedo photon background. A time cut is done for the duration of the GRB to produce the GRB+background data to fit. The time intervals before and after the GRB are used to produce a background model.

In [ ]:
data_path = Path("") # Update to your path

grb_background = UnBinnedData(data_path/'grb.yaml')
grb_background.select_data_time(unbinned_data=data_path/'grb_background.fits.gz', output_name=data_path/'grb_background_source_interval') 
grb_background.get_binned_data(unbinned_data='grb_background_source_interval.fits.gz', output_name='grb_background_binned_galactic', psichi_binning='galactic')
grb_background.load_binned_data_from_hdf5('grb_background_binned_galactic.hdf5')

background_before = UnBinnedData(data_path/'background_before.yaml')
background_before.select_data_time(unbinned_data=data_path/'grb_background.fits.gz', output_name=data_path/'background_before')
background_before.get_binned_data(unbinned_data='background_before.fits.gz', output_name='background_before_binned_galactic', psichi_binning='galactic')
background_before.load_binned_data_from_hdf5('background_before_binned_galactic.hdf5')

background_after = UnBinnedData(data_path/'background_after.yaml') # e.g. background_after.yaml
background_after.select_data_time(unbinned_data=data_path/'grb_background.fits.gz', output_name=data_path/'background_after')
background_after.get_binned_data(unbinned_data='background_after.fits.gz', output_name='background_after_binned_galactic', psichi_binning='galactic')
background_after.load_binned_data_from_hdf5('background_after_binned_galactic.hdf5')

Define the path to the detector response and read in the orientation file. The orientation is cut down to the time interval of the source.

In [ ]:
response_file = data_path/'ResponseContinuum.o3.pol.e200_10000.b4.p12.relx.s10396905069491.m420.filtered.binnedpolarization.11D.h5' # e.g. ResponseContinuum.o3.pol.e200_10000.b4.p12.s10396905069491.m441.filtered.binnedpolarization.11D.h5

sc_orientation = SpacecraftFile.parse_from_file(data_path/'DC3_final_530km_3_month_with_slew_1sbins_GalacticEarth_SAA.ori') # e.g. DC3_final_530km_3_month_with_slew_1sbins_GalacticEarth_SAA.ori
sc_orientation = sc_orientation.source_interval(Time(1835493492.2, format = 'unix'), Time(1835493492.8, format = 'unix'))

Define the GRB position and spectrum.

In [ ]:
source_direction = SkyCoord(l=23.53, b=-53.44, frame='galactic', unit=u.deg)

a = 100. * u.keV
b = 10000. * u.keV
alpha = -0.7368949
beta = -2.095031
ebreak = 622.389 * u.keV
K = 300. / u.cm / u.cm / u.s

spectrum = Band_Eflux(a = a.value,
                      b = b.value,
                      alpha = alpha,
                      beta = beta,
                      E0 = ebreak.value,
                      K = K.value)

spectrum.a.unit = a.unit
spectrum.b.unit = b.unit
spectrum.E0.unit = ebreak.unit
spectrum.K.unit = K.unit

Define initial values of polarization level and angle, fix the spectral parameters to their true values, and create the source model.

In [ ]:
polarization = LinearPolarization(75, 50) # polarization level (percentage out of 100), polarization angle (degrees)
spectral_component = SpectralComponent('grb', spectrum, polarization)

source = PointSource('source',                                 # Name of source (arbitrary, but needs to be unique)
                     l = source_direction.l.deg,               # Longitude (deg)
                     b = source_direction.b.deg,               # Latitude (deg)
                     components = [spectral_component])        # Spectral model

source.components['grb'].shape.K.fix = True
source.components['grb'].shape.E0.fix = True
source.components['grb'].shape.alpha.fix = True
source.components['grb'].shape.beta.fix = True

model = Model(source)

### Polarization fit in ICRS frame

Instantiate the COSI 3ML plugin, combine with the model in a JointLikelihood object, then perform maximum likelihood fit.

In [ ]:
cosi = COSILike("cosi",                                                                                                                                         # COSI 3ML plugin
                dr = response_file,                                                                                                                             # detector response
                data = grb_background.binned_data.project('Em', 'Phi', 'PsiChi'),                                                                               # data (source+background)
                bkg = (background_before.binned_data.project('Em', 'Phi', 'PsiChi') + background_after.binned_data.project('Em', 'Phi', 'PsiChi')) * 0.0016,    # background model 
                sc_orientation = sc_orientation,                                                                                                                # spacecraft orientation
                response_pa_convention = 'RelativeX')                                                                                                           # polarization angle convention of response

plugins = DataList(cosi)

like = JointLikelihood(model, plugins, verbose=False)

like.fit()